# Exportação da camada Gold para CSV

Exporta onze tabelas aprovadas, sem dados pessoais do cadastro, para uma pasta exclusiva por tentativa no Volume autorizado. O manifesto identifica a execução Gold efetiva e só existe quando todos os CSVs foram relidos e reconciliados.

Contrato candidato `gold-csv-v2`: o carregador v1 não aceita este conjunto. A migração e a carga v2 são uma entrega do Backend antes do consumo no portal/IA.


In [ ]:
import json
import os
import re
from datetime import datetime, timezone
from uuid import uuid4
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = "workspace"
SCHEMA_GOLD = "gold"
SCHEMA_VERSION = "gold-csv-v2"
TABELAS_GOLD = [
    "diagnostico_receita", "indicadores_dec_fec", "fila_inspecao", "visao_360_uc",
    "consumo_mensal_uc", "consumo_receita_bairro", "variacao_consumo_uc",
    "faturas_uc", "medidor_uc", "atendimentos_uc", "ocorrencias_uc",
]

dbutils.widgets.text("ambiente", "", "Ambiente (opcional, somente prod)")
dbutils.widgets.text("run_id", "", "Run Gold esperado (opcional)")
ambiente_param = dbutils.widgets.get("ambiente").strip().lower()
run_id_param = dbutils.widgets.get("run_id").strip()
if ambiente_param and ambiente_param != "prod":
    raise RuntimeError("Exportação oficial exige ambiente prod.")

def ler_controle():
    df = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.controle_gold")
    rows = (df.filter(F.col("status_gold") == "CONCLUIDO")
            .orderBy(F.col("processado_em").desc()).limit(2)
            .withColumn("_instante", F.date_format("processado_em", "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX"))
            .collect())
    if not rows or (len(rows) == 2 and rows[0]["processado_em"] == rows[1]["processado_em"]):
        raise RuntimeError("Controle Gold ausente ou ambíguo")
    row = rows[0].asDict()
    required = ["run_id_gold", "run_id_silver", "lote_bronze_silver", "_instante"]
    if any(not isinstance(row.get(k), str) or not row[k].strip() for k in required):
        raise RuntimeError("Controle Gold sem identidade completa")
    if (str(row.get("ambiente", "")).strip().lower() != "prod"
            or row.get("silver_lineage_status") != "CONTROLE_FISICO"
            or row.get("status_silver") != "CONCLUIDO"):
        raise RuntimeError("Controle Gold não comprova publicação produtiva validada")
    generated = [t.strip() for t in str(row.get("tabelas_geradas", "")).split(",")]
    if len(generated) != len(TABELAS_GOLD) or set(generated) != set(TABELAS_GOLD):
        raise RuntimeError("Controle Gold não contempla as onze tabelas contratadas")
    row.pop("processado_em")
    return row

controle_exportacao = ler_controle()
run_id_real = controle_exportacao["run_id_gold"]
if not re.fullmatch(r"[A-Za-z0-9_-]{1,100}", run_id_real):
    raise RuntimeError("Identidade Gold inválida para o destino")
if run_id_param and run_id_param != run_id_real:
    raise RuntimeError("Run solicitado diverge da publicação Gold validada")

def versao_delta(nome):
    return int(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{SCHEMA_GOLD}.{nome} LIMIT 1").first()["version"])

# Job Gold e exportação devem executar em sequência, sem execuções concorrentes.
# A versão de cada tabela é fixada antes de qualquer escrita de CSV.
versoes_delta = {nome: versao_delta(nome) for nome in TABELAS_GOLD}
if ler_controle() != controle_exportacao:
    raise RuntimeError("Publicação Gold mudou antes da exportação")


In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}.gold_export_volume")
PATH_EXPORT_BASE = f"/Volumes/{CATALOG}/{SCHEMA_GOLD}/gold_export_volume/export_csv"
export_id = uuid4().hex
PATH_EXPORT = f"{PATH_EXPORT_BASE}/{run_id_real}/{export_id}"
os.makedirs(PATH_EXPORT, exist_ok=False)


In [ ]:
# Nulo tem representação distinta de string vazia; o marcador literal é reservado.
CSV_OPTIONS = {
    "header": "true", "lineSep": "\n", "sep": ";", "encoding": "UTF-8", "quoteAll": "true", 'multiLine': 'true', 'escape': '"', 'quote': '"',
    "nullValue": "\\N", "emptyValue": "", "dateFormat": "yyyy-MM-dd",
    "timestampFormat": "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX", "timeZone": "UTC",
}
controles_tabelas = {}
for nome in TABELAS_GOLD:
    df = (spark.read.option("versionAsOf", versoes_delta[nome])
          .table(f"{CATALOG}.{SCHEMA_GOLD}.{nome}"))
    for campo in df.schema.fields:
        if campo.dataType.simpleString() == "string" and df.filter(F.col(campo.name) == CSV_OPTIONS["nullValue"]).limit(1).count():
            raise ValueError("Fonte contém marcador reservado de nulo CSV")
    destino = f"{PATH_EXPORT}/{nome}"
    df.write.format("csv").mode("errorifexists").options(**CSV_OPTIONS).save(destino)
    relido = (spark.read.schema(df.schema).options(**CSV_OPTIONS)
              .option("mode", "FAILFAST").option("enforceSchema", "false").csv(destino))
    # Compara o multiconjunto inteiro: conserva duplicidades, decimais e timestamps.
    if df.exceptAll(relido).limit(1).count() or relido.exceptAll(df).limit(1).count():
        raise RuntimeError(f"CSV divergente da Gold: {nome}")
    controles_tabelas[nome] = {
        "registros": df.count(), "versao_delta": versoes_delta[nome],
        "schema": df.schema.jsonValue(),
    }

if ler_controle() != controle_exportacao or any(versao_delta(t) != v for t,v in versoes_delta.items()):
    raise RuntimeError("Gold mudou durante a exportação; tentativa sem manifesto")


## Manifesto da exportação oficial


### Validação e publicação do manifesto

Exige controle produtivo com linhagem física, onze tabelas e run esperado compatível. Fixam-se versões Delta e conferem-se controle/versões ao terminar. Cada tentativa usa diretório exclusivo; falha deixa arquivos sem manifesto e preserva exportações anteriores. Não consumir pastas sem manifesto.

O Job deve serializar Gold → exportação e impedir execuções concorrentes: a promoção Gold de onze tabelas não é uma transação única. Os CSVs são relidos com o schema original e comparados como multiconjuntos antes da liberação. Datas usam ISO; timestamps UTC com microssegundos e offset; decimais preservam escala; `\N` representa nulo e string vazia permanece distinta. O contrato v2 requer adaptação explícita do carregador, sem alterar o snapshot v1.


In [ ]:
# Revalidar imediatamente antes de publicar o manifesto.
if ler_controle() != controle_exportacao or any(versao_delta(t) != v for t,v in versoes_delta.items()):
    raise RuntimeError("Publicação Gold mudou; manifesto não será liberado")
manifesto = {
    "schema_version": SCHEMA_VERSION, "ambiente": "prod",
    "lote": run_id_real, "run_id": run_id_real,
    "run_id_silver": controle_exportacao["run_id_silver"],
    "lote_bronze_silver": controle_exportacao["lote_bronze_silver"],
    "data_processamento": controle_exportacao["_instante"],
    "data_exportacao_csv": datetime.now(timezone.utc).isoformat(),
    "export_id": export_id, "tabelas_exportadas": list(TABELAS_GOLD),
    "controles_tabelas": controles_tabelas, "formato_csv": CSV_OPTIONS,
    "responsavel": "Equipe de Engenharia de Dados (Gold)",
}
caminho_manifesto = os.path.join(PATH_EXPORT, "manifest.json")
conteudo_manifesto = json.dumps(manifesto, indent=2, ensure_ascii=False)
manifesto_temporario = caminho_manifesto + ".tmp"
try:
    with open(manifesto_temporario, "x", encoding="utf-8") as arquivo:
        arquivo.write(conteudo_manifesto)
    os.replace(manifesto_temporario, caminho_manifesto)
except Exception:
    if os.path.exists(manifesto_temporario):
        os.remove(manifesto_temporario)
    raise
print("Exportação validada: onze tabelas e manifesto no diretório desta tentativa.")


## Operação e recuperação

- Executar depois do sucesso do Job Gold, com exclusão de execuções concorrentes. Usar o caminho da tentativa concluída, informado em `PATH_EXPORT`; não misturar arquivos de diretórios diferentes.
- Uma falha antes do manifesto exige uma nova tentativa. Não promover manualmente arquivos parciais nem sobrescrever o manifesto da exportação anterior.
- O run vem exclusivamente de `controle_gold`; o parâmetro opcional somente confirma essa identidade. Não existe fallback manual de lote.
- O manifesto carrega a execução Gold, a origem Silver/Bronze e versões físicas. Os controles e dados reais permanecem no Volume autorizado.
- `gold-csv-v2` é candidato até validar o Job real e integrar migração/carga v2. O Backend v1 deve rejeitá-lo explicitamente; esta entrega não ativa consumidores nem conclui WEB-03.
